[Cell 0] 동적 라이브러리 설치

In [1]:
!pip install -q xgboost scikit-learn numpy

[Cell 1] 라이브러리 및 환경 설정

In [2]:
import os
import requests
import mlflow
import json
import boto3
from ultralytics import YOLO, settings
import mlflow.pytorch

settings.update({"mlflow": False})

# 환경 설정
BACKEND_URL = "http://backend:8000"
MLFLOW_TRACKING_URI = "http://mlflow:5000"
S3_ENDPOINT_URL = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://minio:9000")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# 데이터셋 다운로드 함수
def download_dataset(bucket_name, prefix, local_dir="./data"):
    s3 = boto3.client('s3',
        endpoint_url=S3_ENDPOINT_URL,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY
    )
    if not os.path.exists(local_dir):
        os.makedirs(local_dir)
    print(f"Downloading: {bucket_name}/{prefix} -> {local_dir}")
    paginator = s3.get_paginator('list_objects_v2')
    for result in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if 'Contents' in result:
            for obj in result['Contents']:
                key = obj['Key']
                if key.endswith('/'): continue
                local_file_path = os.path.join(local_dir, os.path.relpath(key, prefix))
                os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
                s3.download_file(bucket_name, key, local_file_path)
    return local_dir

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


[Cell 2] 작업 등록

In [3]:
PROJECT_ID = 1  
DATASET_PATH = "datasets/1D-CNN/Gas-Turbine/v1"  # <-- 버킷 이름/폴더 경로
MODEL_ARCHITECTURE = "XGBoost"

payload = {
    "project_id": PROJECT_ID,
    "model_variant": MODEL_ARCHITECTURE,
    "dataset": DATASET_PATH,
    "params": {"source": "jupyter_notebook"}
}
response = requests.post(f"{BACKEND_URL}/api/v1/jobs/jupyter", json=payload)
response.raise_for_status()
job_info = response.json()

JOB_ID = job_info["id"]
RUN_ID = job_info["run_id"]

print(f"[System] Job Registered! ID: {JOB_ID}, Run ID: {RUN_ID}")

[System] Job Registered! ID: 120, Run ID: a5bf6b9631f94f6f976ee199995ca6f8


[Cell 3] XGBoost 학습 로직

In [4]:
import os
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

with mlflow.start_run(run_id=RUN_ID):
    print("[System] XGBoost Regression Training Started...")
    
    try:
        bucket, prefix = DATASET_PATH.split('/', 1)
        local_data_path = download_dataset(bucket, prefix, local_dir=f"./data/job_{JOB_ID}")
        print(f"[System] Dataset downloaded at: {local_data_path}")

        X = np.load(os.path.join(local_data_path, "X.npy")).astype(np.float32)
        y = np.load(os.path.join(local_data_path, "y.npy")).astype(np.float32)
        print(f"[System] Original X shape: {X.shape}, y shape: {y.shape}")

        if len(X.shape) == 3:
            samples, timesteps, features = X.shape
            X = X.reshape(samples, timesteps * features)
            print(f"[System] Reshaped X for XGBoost: {X.shape}")

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        mlflow.xgboost.autolog()

        print("[System] Model fitting (Regressor)...")
        # Classifier -> Regressor 로 변경
        model = xgb.XGBRegressor(
            n_estimators=100, 
            max_depth=6, 
            learning_rate=0.1,
            tree_method='hist' 
        )
        model.fit(X_train, y_train)

        # 평가 지표 변경 (Accuracy -> MSE, R2 Score)
        preds = model.predict(X_test)
        mse = mean_squared_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        
        mlflow.log_metric("test_mse", mse)
        mlflow.log_metric("test_r2", r2)
        print(f"[System] Test MSE: {mse:.4f}, R2 Score: {r2:.4f}")

        print("[System] Training Complete!")
        status_to_report = "FINISHED"
        message = "XGBoost regression completed successfully."

    except Exception as e:
        print(f"[Error] Training Failed: {e}")
        status_to_report = "FAILED"
        message = str(e)
        mlflow.end_run(status='FAILED')

[System] XGBoost Regression Training Started...
Downloading: datasets/1D-CNN/Gas-Turbine/v1 -> ./data/job_120
[System] Dataset downloaded at: ./data/job_120
[System] Original X shape: (36733, 1, 9), y shape: (36733, 2)
[System] Reshaped X for XGBoost: (36733, 9)
[System] Model fitting (Regressor)...


2026/03/25 03:46:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 03:46:16 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'


[System] Test MSE: 11.2064, R2 Score: 0.8073
[System] Training Complete!
🏃 View run sincere-ram-452 at: http://mlflow:5000/#/experiments/4/runs/a5bf6b9631f94f6f976ee199995ca6f8
🧪 View experiment at: http://mlflow:5000/#/experiments/4


[Cell 4] Webhook 전송

In [5]:
webhook_url = f"{BACKEND_URL}/api/v1/jobs/{JOB_ID}/complete"
resp = requests.post(webhook_url, json={"status": status_to_report, "message": message})

if resp.status_code == 200:
    print(f"[System] Webhook sent successfully. Job {JOB_ID} is now {status_to_report}.")
else:
    print(f"[Error] Webhook failed: {resp.status_code}")

[System] Webhook sent successfully. Job 120 is now FINISHED.
